# Longform TTS, 긴 텍스트 자동 청킹
- 30초 넘는 긴 텍스트는 한 번에 합성하면 메모리 부담·품질 저하·오류 가능성이 큽니다. OmniVoice 의 `audio_chunk_duration` / `audio_chunk_threshold` 인자가 **자동으로 텍스트를 청킹** 해서 안정적 합성 + 일정한 VRAM 으로 끝까지 처리합니다.


## 0. 셋업

In [2]:
# 라이브러리 다운로드
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.0/163.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.4 MB/s eta 0:00:00


- Hugging Face 토큰 에러 발생시 아래 코드 실행

In [ ]:
# import os

# os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# try:
#     import huggingface_hub.constants as hf_constants
#     hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
# except Exception:
#     pass

In [3]:
import torch
import soundfile as sf
import numpy as np
from omnivoice import OmniVoice
from IPython.display import Audio
import os
os.makedirs("outputs", exist_ok=True)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)
print("준비 완료. device:", device)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

준비 완료. device: cuda:0


## 1. 청킹 파라미터 이해

| 파라미터 | 기본값 | 의미 |
|---|---|---|
| `audio_chunk_duration` | 15.0 | 청크 한 개의 목표 길이 (초) |
| `audio_chunk_threshold` | 30.0 | 예상 음성이 이 길이를 넘으면 청킹 발동 |

짧은 텍스트는 청킹 안 됨 (한 번에). 긴 텍스트는 자동으로 ~15초 단위로 잘려서 순차 합성 후 이어붙임. VRAM 은 청크 하나 분량으로 거의 일정.

## 2. 짧은 텍스트, 청킹 발동 안 됨

In [4]:
short = "안녕하세요. 학생 여러분 반갑습니다."
audio = model.generate(text=short)
sf.write("outputs/long_short.wav", audio[0], 24000)
print(f"길이: {len(audio[0])/24000:.1f}초")
Audio("outputs/long_short.wav")

길이: 2.2초


## 3. 긴 문단, 자동 청킹

오디오북·강의 내레이션 같은 1분 이상 분량. `audio_chunk_threshold=30` 기본값을 넘으니 자동으로 ~15초 단위 청킹.

In [7]:
long_text = (
    "SK 네트웍스 패밀리 AI 캠프에 오신 것을 환영합니다. "
    "오늘 우리가 다루는 주제는 OmniVoice 라는 음성 합성 모델입니다. "
    "이 모델은 600개 이상의 언어를 지원하며, 짧은 참조 음성만으로 누구의 목소리든 따라할 수 있는 zero-shot 보이스 클로닝 기능을 갖추고 있습니다. "
    "또한 화자 속성을 자연어로 지정하는 Voice Design, 비언어 감정 표현 태그, 다중 화자 대화 같은 다양한 활용이 가능합니다. "
    "이번 단원에서는 짧은 합성부터 시작해서, 한국어 합성, 보이스 클로닝, 그리고 지금처럼 긴 문단을 한 번에 합성하는 Longform TTS 까지 차근차근 익혀봅시다. "
    "준비되셨으면 다음 셀로 넘어가서 직접 합성을 시도해봅시다."
    "한국인도 겁내는 음식” 뭐길래…“오마이갓” 한입 먹고 혼쭐났다"
    "새벽까지 수도권 120㎜ 물폭탄…비 그친 뒤엔 ‘찜통 더위’"
)

import time
t0 = time.time()
audio = model.generate(text=long_text)
dt = time.time() - t0

duration_sec = len(audio[0]) / 24000
print(f"생성 음성 길이 : {duration_sec:.1f}초")
print(f"합성 소요 시간 : {dt:.1f}초")
# RTF = 합성 소요 시간 / 생성된 음성 길이.
# TTS 모델의 추론 속도 성능을 평가하는 값.
# RTF = 1.0 -> 실시간 수준 / RTF = 2.0 -> 2배 느림 / RTF = 0.5 -> 2배 빠름
print(f"RTF (real-time factor): {dt/duration_sec:.3f}  ( <1 이면 실시간보다 빠름 )")

sf.write("outputs/long_paragraph.wav", audio[0], 24000)
Audio("outputs/long_paragraph.wav")

Output hidden; open in https://colab.research.google.com to view.

## 4. 청킹 파라미터 조절

청크가 짧을수록 VRAM 사용 적지만 청크 사이 톤이 살짝 끊길 수 있음. 길수록 자연스럽지만 메모리 부담 ↑.

In [8]:
# 청크 짧게: 8초 단위 (VRAM 우선)
audio_short_chunk = model.generate(
    text=long_text,
    audio_chunk_duration=8.0,           # 약 8초 단위로 나눠 합성
    audio_chunk_threshold=15.0          # 생성될 음성이 약 15초를 넘든다고 판단되면 청킹
)
sf.write("outputs/long_chunk8s.wav", audio_short_chunk[0], 24000)

# 청크 길게: 25초 단위 (자연스러움 우선)
audio_long_chunk = model.generate(
    text=long_text,
    audio_chunk_duration=25.0,
    audio_chunk_threshold=50.0
)
sf.write("outputs/long_chunk25s.wav", audio_long_chunk[0], 24000)

print("두 결과 비교 (이어듣기)")
display(Audio("outputs/long_chunk8s.wav"))
display(Audio("outputs/long_chunk25s.wav"))


Output hidden; open in https://colab.research.google.com to view.

## 5. Longform + Voice Cloning

긴 내레이션을 본인 목소리로. `samples/my_voice.wav` 재활용.

In [10]:
if os.path.exists("my_voice.wav"):
    REF_TEXT = "안녕하세요. 제 이름은 안현준입니다. 인공지능 강의를 맡고 있습니다. 오늘도 화이팅입니다."
    audio_clone_long = model.generate(
        text=long_text,
        ref_audio="my_voice.wav",
        ref_text=REF_TEXT
    )

    sf.write("outputs/long_clone.wav", audio_clone_long[0], 24000)
    print(f"길이: {len(audio_clone_long[0])/24000:.1f}초")
    display(Audio("outputs/long_clone.wav"))
else:
    print("samples/my_voice.wav 가 없습니다. 먼저 녹음하세요.")

Output hidden; open in https://colab.research.google.com to view.

## 6. Longform + Voice Design

캐릭터별 톤으로 긴 내레이션. NPC 가 긴 배경 스토리를 들려주는 시나리오.

In [11]:
audio_design_long = model.generate(
    text=long_text,
    instruct="male, middle-aged, low pitch"
)

sf.write("outputs/long_design.wav", audio_design_long[0], 24000)
Audio("outputs/long_design.wav")

Output hidden; open in https://colab.research.google.com to view.

## [실습]
1. `audio_chunk_duration` 을 5 / 15 / 30 으로 바꿔보고 결과 자연스러움 비교.
2. 1분짜리 강의 도입부 텍스트를 본인 목소리 (Voice Cloning) 로 합성
3. 같은 긴 텍스트를 Voice Design 두 톤 (예: 진중한 내레이터 vs 발랄한 진행자) 으로 합성해 비교
4. Longform TTS 한계 찾기(어떤 입력에서 품질이 떨어지는지)